In [1]:
# 04_counterfactual.ipynb
# =============================================================================
# Personalized Health Intervention Pathways for Comorbid Policyholders:
# Loss Ratio Management via LLM Guardrails and Counterfactual Explanations
# -----------------------------------------------------------------------------
# Notebook 04 — Guardrail-Constrained DiCE + Stepwise Fallback (Multi-Case)
# Input  : ../results/tables/agent_config.pkl
#          ../results/tables/df_final.pkl
#          ../results/tables/model_{group}.pkl
#          ../results/tables/guardrail_ranges.json
# Output : ../results/tables/counterfactuals.json
#          ../results/tables/cf_summary.csv
# =============================================================================

# %%
# ## 0. Import Libraries

import json
import joblib
import warnings
import numpy as np
import pandas as pd
import dice_ml

warnings.filterwarnings('ignore')

# %%
# ## 1. Load Models, Configuration, and Guardrail Ranges

agent_config  = joblib.load('../results/tables/agent_config.pkl')
df_final      = joblib.load('../results/tables/df_final.pkl')

X_FEATURES      = agent_config['X_features']
CAT_FEATURES    = agent_config['cat_features']
VARY_FEATURES   = agent_config['vary_features']
FIXED_FEATURES  = agent_config['fixed_features']
TARGET_COL      = agent_config['target_col']
AGEGROUP_CONFIG = agent_config['agegroup_config']

models = {}
for group_name in AGEGROUP_CONFIG:
    try:
        models[group_name] = joblib.load(
            f'../results/tables/model_{group_name}.pkl'
        )
    except FileNotFoundError:
        print(f"WARNING: model_{group_name}.pkl not found.")
        models[group_name] = None

with open('../results/tables/guardrail_ranges.json', 'r', encoding='utf-8') as f:
    all_guardrails = json.load(f)

print("Models loaded   :", [k for k, v in models.items() if v is not None])
print("Guardrail cases :", list(all_guardrails.keys()))

# %%
# ## 2. Post-Processing: Anthropometric Coupling Enforcement

def enforce_anthro_coupling(cf_df: pd.DataFrame,
                             orig_bmi: float,
                             orig_waist: float,
                             orig_wt: float) -> pd.DataFrame:
    """
    Row-wise enforcement of Rule A coupling after DiCE search.
    Aligns BMI, WaistCirc, and Weight to the largest reduction ratio
    so all three move in the same direction and at a consistent ratio.
    """
    def _fix(row):
        b, w, wt = row['BMI'], row['WaistCirc'], row['Weight']
        if (abs(b  - orig_bmi)   < 0.01 and
            abs(w  - orig_waist) < 0.01 and
            abs(wt - orig_wt)    < 0.01):
            return row
        r_target = max(0.85, min(b/orig_bmi, w/orig_waist, wt/orig_wt, 1.0))
        if b  / orig_bmi   > r_target: row['BMI']       = round(orig_bmi   * r_target, 4)
        if w  / orig_waist > r_target: row['WaistCirc'] = round(orig_waist * r_target, 4)
        if wt / orig_wt    > r_target: row['Weight']    = round(orig_wt    * r_target, 4)
        return row

    return cf_df.apply(_fix, axis=1)

# %%
# ## 3. Categorical Variable Snapping

def snap_categoricals(cf_df: pd.DataFrame,
                       cat_features: list,
                       fixed_features: list,
                       df_ref: pd.DataFrame) -> pd.DataFrame:
    """
    Round DiCE continuous outputs to the nearest valid integer
    for categorical features that are not fixed.
    """
    vary_cat = [f for f in cat_features if f not in fixed_features]
    for col in vary_cat:
        if col not in cf_df.columns:
            continue
        valid_vals = sorted(df_ref[col].dropna().unique().tolist())
        cf_df[col] = cf_df[col].apply(
            lambda v: min(valid_vals, key=lambda x: abs(x - v))
        )
    return cf_df

# %%
# ## 4. Clinical Constraint Clipping (Post-Processing)

def apply_clinical_constraints(cf_df: pd.DataFrame,
                                hard_ranges: dict,
                                orig_bmi: float,
                                orig_waist: float,
                                orig_wt: float) -> pd.DataFrame:
    """
    Clip each CF candidate to hard rule boundaries after DiCE search,
    then enforce anthropometric coupling.
    """
    for feat, (lo, hi) in hard_ranges.items():
        if feat in cf_df.columns:
            cf_df[feat] = cf_df[feat].clip(lower=lo, upper=hi)
    cf_df = enforce_anthro_coupling(cf_df, orig_bmi, orig_waist, orig_wt)
    return cf_df

# %%
# ## 5. Search Range Builder (Wide + Hard Floor)

def build_search_range(hard_ranges: dict,
                        df_ref: pd.DataFrame,
                        features: list) -> dict:
    """
    Build DiCE search range by combining hard rule floors with
    data-driven 5th–95th percentile range.
    This ensures sufficient Class 0 samples exist in the search space.
    Clinical constraints are applied as post-processing (Section 4).
    """
    search_range = {}
    for feat in features:
        hard_lo, hard_hi = hard_ranges.get(feat, [0, float('inf')])
        data_lo = float(df_ref[feat].quantile(0.05))
        data_hi = float(df_ref[feat].quantile(0.95))
        lo = max(min(hard_lo, data_lo), 0.0)
        hi = max(hard_hi, data_hi)
        search_range[feat] = [round(lo, 4), round(hi, 4)]
    return search_range

# %%
# ## 6. Guardrail-Constrained DiCE Pipeline with Stepwise Fallback

def run_dice_pipeline(model,
                      df_input: pd.DataFrame,
                      age_group: float,
                      gender_code: float,
                      case_key: str,
                      guardrails: dict) -> tuple:
    """
    Run guardrail-constrained DiCE with stepwise fallback.

    Search space      : data-driven 5–95 pct range (wide)
    Clinical guardrail: applied as post-processing clip
    Fallback order    : Class 0 → Class 1 (HTN) → Class 2 (DM)

    Returns
    -------
    (cf_df, achieved_class, fallback_depth)
    """
    print(f"\n{'='*20} [{case_key}] DiCE Pipeline {'='*20}")

    df_stable = df_input.copy().astype(float)
    df_ref    = df_stable[
        (df_stable['AgeGroup'] == age_group) &
        (df_stable['Sex']      == gender_code)
    ].copy()

    profile     = guardrails['patient_profile']
    hard_ranges = guardrails['final_ranges']
    query       = pd.DataFrame([profile])[X_FEATURES]

    orig_bmi   = float(query.iloc[0]['BMI'])
    orig_waist = float(query.iloc[0]['WaistCirc'])
    orig_wt    = float(query.iloc[0]['Weight'])

    print(f"Patient — BMI: {orig_bmi:.2f}, "
          f"Energy: {float(query.iloc[0]['Energy_kcal']):.1f} kcal, "
          f"Sodium: {float(query.iloc[0]['Sodium_mg']):.1f} mg")
    print(f"Predicted class: {int(model.predict(query)[0])}")

    search_range = build_search_range(hard_ranges, df_ref, X_FEATURES)

    # DiCE setup
    d   = dice_ml.Data(
            dataframe=df_stable[X_FEATURES + [TARGET_COL]],
            continuous_features=X_FEATURES,
            outcome_name=TARGET_COL,
          )
    m   = dice_ml.Model(model=model, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="genetic")

    fallback_map = {0: "Full normalisation",
                    1: "HTN-priority",
                    2: "DM-priority"}

    for target_class, target_name in fallback_map.items():
        print(f"\n▶ Targeting Class {target_class} ({target_name})...")

        for attempt in range(15):
            try:
                cf = exp.generate_counterfactuals(
                    query,
                    total_CFs=4,
                    desired_class=target_class,
                    features_to_vary=VARY_FEATURES,
                    permitted_range=search_range,
                    proximity_weight=0.2,
                    sparsity_weight=0.1,
                )
                if cf and len(cf.cf_examples_list[0].final_cfs_df) > 0:
                    cf_df = cf.cf_examples_list[0].final_cfs_df.copy()
                    cf_df = snap_categoricals(
                        cf_df, CAT_FEATURES, FIXED_FEATURES, df_stable
                    )
                    cf_df = apply_clinical_constraints(
                        cf_df, hard_ranges, orig_bmi, orig_waist, orig_wt
                    )
                    print(f"  ✓ Class {target_class} reached "
                          f"(attempt {attempt + 1})")
                    return cf_df, target_class, target_class

            except Exception:
                continue

        print(f"  ✗ Class {target_class} unreachable. Falling back...")

    print("  ✗ All targets exhausted.")
    return None, None, None

# %%
# ## 7. Run Pipeline for All 12 Cases

cf_results = {}

for case_key, guardrail_data in all_guardrails.items():
    group_name  = guardrail_data['group']
    mdl         = models.get(group_name)
    cfg         = AGEGROUP_CONFIG[group_name]
    age_grp     = 0.0 if cfg['age_min'] < 60 else 1.0

    if mdl is None:
        print(f"\n[{case_key}] Model not available. Skipping.")
        cf_results[case_key] = (None, None, None)
        continue

    cf_df, achieved_class, fallback_depth = run_dice_pipeline(
        model       = mdl,
        df_input    = df_final,
        age_group   = age_grp,
        gender_code = cfg['sex_code'],
        case_key    = case_key,
        guardrails  = guardrail_data,
    )
    cf_results[case_key] = (cf_df, achieved_class, fallback_depth)

# %%
# ## 8. Display Results Summary

DISPLAY_VARS = ['BMI', 'WaistCirc', 'Weight',
                'Energy_kcal', 'Carb_g', 'Sugar_g',
                'Sodium_mg', 'Fat_g']

fallback_map = {0: "Full normalisation",
                1: "HTN-priority",
                2: "DM-priority"}

for case_key, (cf_df, achieved_class, fallback_depth) in cf_results.items():
    print(f"\n{'='*15} [{case_key}] {'='*15}")
    if cf_df is None:
        print("  No valid recourse found.")
        continue

    print(f"  Achieved : Class {achieved_class} "
          f"({fallback_map.get(achieved_class, '?')})"
          + (" [FALLBACK]" if fallback_depth > 0 else ""))

    profile  = all_guardrails[case_key]['patient_profile']
    orig_row = pd.DataFrame([profile])[X_FEATURES]

    n_cfs = len(cf_df)
    header = f"  {'Variable':20s} | {'Original':>10s} | " + \
             " | ".join([f"{'CF'+str(i+1):>10s}" for i in range(n_cfs)])
    print(header)
    print("  " + "-" * len(header))

    for var in DISPLAY_VARS:
        if var not in orig_row.columns:
            continue
        orig_val = float(orig_row.iloc[0][var])
        cf_vals  = [float(cf_df.iloc[i][var]) for i in range(n_cfs)]
        print(f"  {var:20s} | {orig_val:>10.2f} | " +
              " | ".join([f"{v:>10.2f}" for v in cf_vals]))

# %%
# ## 9. Save Results

export = {
    "metadata": {
        "model":          "XGBoost + GPT-4o-mini guardrail + DiCE",
        "data":           "KNHANES 2020-2024",
        "stratification": "AgeGroup x Sex (4 groups, 3 cases each)",
        "total_CFs":      4,
        "total_cases":    len(cf_results),
    },
    "results": [],
}

summary_records = []

for case_key, (cf_df, achieved_class, fallback_depth) in cf_results.items():
    guardrail_data = all_guardrails[case_key]
    if cf_df is not None:
        entry = {
            "case_key":       case_key,
            "group":          guardrail_data['group'],
            "case":           guardrail_data['case'],
            "status":         "success",
            "achieved_class": int(achieved_class),
            "fallback_depth": int(fallback_depth),
            "original":       guardrail_data['patient_profile'],
            "counterfactuals": cf_df.to_dict(orient='records'),
        }
        summary_records.append({
            "CaseKey":       case_key,
            "Group":         guardrail_data['group'],
            "Case":          guardrail_data['case'],
            "AchievedClass": achieved_class,
            "FallbackDepth": fallback_depth,
            "Status":        "success",
        })
    else:
        entry = {"case_key": case_key, "status": "failed"}
        summary_records.append({
            "CaseKey":       case_key,
            "Group":         guardrail_data['group'],
            "Case":          guardrail_data['case'],
            "AchievedClass": None,
            "FallbackDepth": None,
            "Status":        "failed",
        })
    export["results"].append(entry)

with open('../results/tables/counterfactuals.json', 'w', encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False, indent=4)

summary_df = pd.DataFrame(summary_records)
summary_df.to_csv('../results/tables/cf_summary.csv',
                  index=False, encoding='utf-8-sig')

print("\n>>> Saved: ../results/tables/counterfactuals.json")
print(">>> Saved: ../results/tables/cf_summary.csv")
print(summary_df.to_string(index=False))

Models loaded   : ['MiddleAged_Male', 'MiddleAged_Female', 'Older_Male', 'Older_Female']
Guardrail cases : ['MiddleAged_Male_case1', 'MiddleAged_Male_case2', 'MiddleAged_Male_case3', 'MiddleAged_Female_case1', 'MiddleAged_Female_case2', 'MiddleAged_Female_case3', 'Older_Male_case1', 'Older_Male_case2', 'Older_Male_case3', 'Older_Female_case1', 'Older_Female_case2', 'Older_Female_case3']

==================== [MiddleAged_Male_case1] DiCE Pipeline ====================
Patient — BMI: 27.59, Energy: 1934.6 kcal, Sodium: 2925.9 mg
Predicted class: 3

▶ Targeting Class 0 (Full normalisation)...


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.62it/s]


  ✓ Class 0 reached (attempt 1)

==================== [MiddleAged_Male_case2] DiCE Pipeline ====================
Patient — BMI: 26.99, Energy: 1892.1 kcal, Sodium: 4818.4 mg
Predicted class: 3

▶ Targeting Class 0 (Full normalisation)...


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.22it/s]


  ✓ Class 0 reached (attempt 1)

==================== [MiddleAged_Male_case3] DiCE Pipeline ====================
Patient — BMI: 25.03, Energy: 1886.4 kcal, Sodium: 3780.3 mg
Predicted class: 3

▶ Targeting Class 0 (Full normalisation)...


...

  ✓ Class 0 reached (attempt 1)

==================== [Older_Female_case2] DiCE Pipeline ====================
Patient — BMI: 26.27, Energy: 1344.8 kcal, Sodium: 2220.4 mg
Predicted class: 3

▶ Targeting Class 0 (Full normalisation)...


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.65it/s]


  ✓ Class 0 reached (attempt 1)

==================== [Older_Female_case3] DiCE Pipeline ====================
Patient — BMI: 26.06, Energy: 1341.9 kcal, Sodium: 1790.8 mg
Predicted class: 3

▶ Targeting Class 0 (Full normalisation)...


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.60it/s]

  ✓ Class 0 reached (attempt 1)

=============== [MiddleAged_Male_case1] ===============
  Achieved : Class 0 (Full normalisation)
  Variable             |   Original |        CF1 |        CF2 |        CF3 |        CF4
  ---------------------------------------------------------------------------------------
  BMI                  |      27.59 |      23.45 |      23.45 |      23.45 |      27.59
  WaistCirc            |      99.20 |      84.32 |      84.32 |      84.32 |      99.20
  Weight               |      84.10 |      71.48 |      71.48 |      71.48 |      84.10
  Energy_kcal          |    1934.58 |    1917.52 |    1934.58 |    1681.40 |    1934.58
  Carb_g               |     237.05 |     209.11 |     237.05 |     140.01 |     237.05
  Sugar_g              |      32.42 |      32.42 |      32.42 |      32.42 |      32.42
  Sodium_mg            |    2925.93 |    2633.34 |    2633.34 |    2633.34 |    2633.34
  Fat_g                |      69.64 |      48.75 |      73.99 |      81.98 